In [4]:
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModel
import torch

In [5]:
mind_news_raw = pd.read_csv('/Users/harshadayiniakula/Desktop/RS/MINDsmall_dev/news.tsv', sep='\t', header=None)
print(mind_news_raw.shape)
mind_news_raw.head()

(42416, 8)


,0,1,2,3,4,5,6,7
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N18955,health,medical,Dispose of unwanted prescription drugs during ...,NaN,https://assets.msn.com/labs/mind/AAISxPN.html,"[{""Label"": ""Drug Enforcement Administration"", ...",[]
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


Cleaning the data

In [10]:
import os

In [12]:
news = pd.read_table(os.path.join(data_path, 'train', 'news.tsv'),
                     names=['newid', 'vertical', 'subvertical', 'title',
                            'abstract', 'url', 'entities in title', 'entities in abstract'],
                     usecols = ['vertical', 'subvertical', 'title', 'abstract'])

NameError: name 'data_path' is not defined

In [14]:
#reading the csv into pandas and assiging column names to each column
mind_news = pd.read_csv(
    '/Users/harshadayiniakula/Desktop/RS/MINDsmall_dev/news.tsv',
    sep='\t',
    header=None,
    names=['newid', 'vertical', 'subvertical', 'title', 'abstract', 'url', 'entities in title', 'entities in abstract'],
    dtype=str
)

In [16]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text.strip()

In [18]:
mind_news['title_clean'] = mind_news['title'].fillna('').apply(clean)
mind_news['abstract_clean'] = mind_news['abstract'].fillna('').apply(clean)

In [20]:
#making a full text column by combining both title and abstract columns into one
mind_news['full_text'] = mind_news['title_clean'] + ' ' + mind_news['abstract_clean']

In [22]:
mind_news[['newid','title_clean', 'abstract', 'full_text']].head(3)


,newid,title_clean,abstract,full_text
0,N55528,the brands queen elizabeth prince charles and ...,"Shop the notebooks, jackets, and more that the...",the brands queen elizabeth prince charles and ...
1,N18955,dispose of unwanted prescription drugs during ...,NaN,dispose of unwanted prescription drugs during ...
2,N61837,the cost of trumps aid freeze in the trenches ...,Lt. Ivan Molchanets peeked over a parapet of s...,the cost of trumps aid freeze in the trenches ...


In [24]:
# Load the multilingual model
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [25]:
# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(250037, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

In [28]:
# Prepare input texts
english_texts = mind_news['full_text'].tolist()
batch_size = 64
english_embeddings = []

In [30]:
with torch.no_grad():
    for start in range(0, len(english_texts), batch_size):
        end = start + batch_size
        batch = english_texts[start:end]

        # Tokenize batch
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Model forward pass
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state

        # Mean pooling
        attention = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
        summed = torch.sum(last_hidden * attention, dim=1)
        counts = torch.clamp(attention.sum(dim=1), min=1e-9)
        batch_embeddings = (summed / counts).cpu()

        english_embeddings.append(batch_embeddings)


In [32]:
english_embeddings = torch.cat(english_embeddings, dim=0)
print(f"English article embeddings created: {english_embeddings.shape}")

English article embeddings created: torch.Size([42416, 384])


In [36]:
torch.save(english_embeddings, 'english_article_embeddings_val.pt')

In [38]:
import pickle
with open('english_news_ids_val.pkl', 'wb') as f:
    pickle.dump(mind_news['newid'].tolist(), f)